# PKG Attrition — Source Data EDA

**Payment Knowledge Graph (PKG) · PNC Treasury Management · Data Science**

Read-only diagnostic pass over the Neo4j staging payments table and the deposit
daily table, to answer the mechanics questions that gate the attrition study.

---

**Design note — output density.** This notebook is built to produce **eight
screenshot-able output cells**, not eighty little dataframes. Each numbered
section ends in one consolidated block with an auto-computed `VERDICT` line, so
a single image carries both the evidence and the conclusion.

| Output | Section | Answers |
|---|---|---|
| 1 | Source census | table shapes, null profile, `deposit_family` inventory, rail × category inventory — **sets the filters for everything below** |
| 2 | Deposit panel grain | one-row-per-account-per-day? duplicate groups shown transposed, all columns; business-day completeness; per-account gaps |
| 3 | Status semantics | is `acct_status` / `closed_dt` as-of, or current-state stamped on history? rows and balances after closure |
| 4 | Closure definitions | five candidate definitions vs. the real flag: agreement matrix and timing lead/lag |
| 5 | `avg_monthly_bal_1` | MTD / prior-full-month / current-full-month / trailing-30d — which one is it |
| 6 | Payments profile | leg classification, amounts, `cpty_name` and `cpty_fin_entity_name` coverage, one-row-per-transaction verification, first read on exact name match |
| 7 | Join coverage | `pnc_dep_acct_*` presence and join rate into deposits; `cust_pwr_id` ↔ `mdm_id` cardinality |
| 8 | Episode feasibility | closures per month, history depth, event-time balance curve for closers vs controls |

**Two-pass by design.** Run §1 first, read the `deposit_family` inventory, set
`DEPOSIT_FAMILY_KEEP` in the config, then run §2 onward. The regex default is a
guess and should not be trusted.

**No writes.** Temp views and `persist()` on narrow frames only.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  §0 · CONFIG
# ═══════════════════════════════════════════════════════════════════════════
PAYMENTS_TBL = "<db>.neo4j_payments"          # <-- FILL IN
DEPOSITS_TBL = "<db>.<deposit_daily_table>"   # <-- FILL IN

DATE_START = "2024-01-01"
DATE_END   = "2026-07-31"

# Date columns may be DATE or STRING 'yyyy-MM-dd'; the cast handles both.
DEP_DT_EXPR = "TO_DATE(CAST(edw_tda_load_dt AS STRING))"
PAY_DT_EXPR = "TO_DATE(CAST(trans_dt AS STRING))"

# --- set AFTER reading Output 1 -------------------------------------------
DEPOSIT_FAMILY_KEEP  = None                                   # e.g. ["DDA", "MMDA"]
DEPOSIT_FAMILY_REGEX = r"(?i)(DDA|DEMAND|MMDA|MONEY[ _]*MARKET)"   # fallback guess
# --------------------------------------------------------------------------

SEMANTICS_SAMPLE_ACCTS = 20_000   # §5 and §2 dup-detail run on a sample; 20k is ample
ZERO_BAL_TOL           = 100.0    # $ below which a balance counts as "empty"
ZERO_BAL_MONTHS        = 3        # consecutive empty months for the balance-based closure
DISAPPEAR_GRACE_DAYS   = 45       # last-seen older than this before panel end = disappeared
DROP_RULE_RATIO        = 0.70     # current rule: trailing-3m avg < 0.70 x prior-6m avg
EVENT_WINDOW           = 12       # months of history to draw in the event-time curve
CONTROL_SAMPLE         = 20_000   # non-closing accounts sampled for the control curve
SEED                   = 20260903

In [ ]:
import time, math, re, json
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession, functions as F, Window as W

try:
    spark
except NameError:
    spark = SparkSession.builder.appName("pkg_attrition_source_eda").getOrCreate()

spark.conf.set("spark.sql.shuffle.partitions", "400")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_colwidth", 44)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}" if abs(v) < 1e6 else f"{v:,.0f}")

_T0 = time.time()


def rule(title, ch="=", width=118):
    print(f"\n{ch * width}\n  {title}\n{ch * width}")


def sub(title, width=118):
    pad = max(0, width - len(title) - 5)
    print(f"\n-- {title} " + "-" * pad)


def show(obj, n=None, index=False):
    if isinstance(obj, pd.DataFrame):
        p = obj if n is None else obj.head(n)
        print(p.to_string(index=index) if len(p) else "   (empty)")
    elif isinstance(obj, pd.Series):
        print(obj.to_string())
    else:
        print(obj)


def verdict(*lines):
    print("\n" + "*" * 118)
    for ln in lines:
        print(f"  VERDICT  {ln}")
    print("*" * 118)


def pct(a, b):
    return 100.0 * a / b if b else float("nan")


def nzs(col):
    """Trimmed string, with '' collapsed to NULL. Single absent representation."""
    s = F.trim(F.col(col).cast("string"))
    return F.when(s.isNull() | (s == ""), None).otherwise(s)


def elapsed():
    m, s = divmod(time.time() - _T0, 60)
    return f"[{int(m):02d}:{int(s):02d}]"


print(f"{elapsed()} config loaded | window {DATE_START} .. {DATE_END}")

In [ ]:
# ── Base views. Date + window filters pushed in; nothing else assumed. ──────
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW dep_raw AS
SELECT *, {DEP_DT_EXPR} AS dt
FROM   {DEPOSITS_TBL}
WHERE  {DEP_DT_EXPR} BETWEEN DATE'{DATE_START}' AND DATE'{DATE_END}'
""")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW pay_raw AS
SELECT *, {PAY_DT_EXPR} AS dt
FROM   {PAYMENTS_TBL}
WHERE  {PAY_DT_EXPR} BETWEEN DATE'{DATE_START}' AND DATE'{DATE_END}'
""")

dep_raw = spark.table("dep_raw")
pay_raw = spark.table("pay_raw")
print(f"{elapsed()} views created")
print("dep_raw columns:", ", ".join(dep_raw.columns))
print("pay_raw columns:", ", ".join(pay_raw.columns))

---
## OUTPUT 1 · Source census

Shapes, null/blank profile, and the two value inventories that set every filter
downstream. **Read `deposit_family` here and set `DEPOSIT_FAMILY_KEEP` before
running §2.**

In [ ]:
def null_profile(sdf, name):
    n = sdf.count()
    aggs = []
    for c in sdf.columns:
        s = F.trim(F.col(c).cast("string"))
        aggs.append(F.sum(F.when(F.col(c).isNull() | (s == ""), 1).otherwise(0)).alias(f"n__{c}"))
        aggs.append(F.approx_count_distinct(F.coalesce(s, F.lit("\x00NULL")), 0.02).alias(f"d__{c}"))
    r = sdf.agg(*aggs).collect()[0].asDict()
    out = pd.DataFrame({
        "column":      sdf.columns,
        "null_blank":  [r[f"n__{c}"] for c in sdf.columns],
        "approx_ndv":  [r[f"d__{c}"] for c in sdf.columns],
    })
    out.insert(2, "null_pct", (100.0 * out.null_blank / max(n, 1)).round(2))
    out["table"] = name
    return n, out


rule("OUTPUT 1 · SOURCE CENSUS")

n_dep, dep_nulls = null_profile(dep_raw, "deposits")
n_pay, pay_nulls = null_profile(pay_raw, "payments")

dep_span = dep_raw.agg(
    F.min("dt").alias("min_dt"), F.max("dt").alias("max_dt"),
    F.countDistinct("dt").alias("n_dates"),
    F.approx_count_distinct("acct_full_acct_id", 0.01).alias("approx_accts"),
).toPandas()
pay_span = pay_raw.agg(
    F.min("dt").alias("min_dt"), F.max("dt").alias("max_dt"),
    F.countDistinct("dt").alias("n_dates"),
    F.approx_count_distinct("trans_id", 0.01).alias("approx_trans"),
).toPandas()

sub("shape")
shape = pd.DataFrame([
    {"table": "deposits", "rows": n_dep, "min_dt": dep_span.min_dt[0], "max_dt": dep_span.max_dt[0],
     "n_distinct_dates": dep_span.n_dates[0], "approx_keys": dep_span.approx_accts[0], "key": "acct_full_acct_id"},
    {"table": "payments", "rows": n_pay, "min_dt": pay_span.min_dt[0], "max_dt": pay_span.max_dt[0],
     "n_distinct_dates": pay_span.n_dates[0], "approx_keys": pay_span.approx_trans[0], "key": "trans_id"},
])
show(shape)

sub("null / blank profile  (blank string counted as null)")
show(pd.concat([dep_nulls, pay_nulls])[["table", "column", "null_blank", "null_pct", "approx_ndv"]])

sub("deposit_family inventory  <-- SET DEPOSIT_FAMILY_KEEP FROM THIS")
fam = (dep_raw.groupBy(nzs("deposit_family").alias("deposit_family"))
       .agg(F.countDistinct("acct_full_acct_id").alias("n_accts"),
            F.count("*").alias("n_rows"),
            F.avg("balance").alias("mean_bal"),
            F.expr("percentile_approx(balance, 0.5)").alias("median_bal"),
            F.sum("balance").alias("sum_bal_all_days"))
       .orderBy(F.desc("n_accts")).toPandas())
fam["regex_hit"] = fam.deposit_family.fillna("").str.contains(DEPOSIT_FAMILY_REGEX, regex=True)
fam["acct_pct"] = (100 * fam.n_accts / fam.n_accts.sum()).round(2)
show(fam[["deposit_family", "n_accts", "acct_pct", "n_rows", "mean_bal", "median_bal", "regex_hit"]])

sub("payment_rail x category inventory")
rc = (pay_raw.groupBy(nzs("payment_rail").alias("payment_rail"), nzs("category").alias("category"))
      .agg(F.count("*").alias("n_txn"),
           F.sum("trans_amt").alias("sum_amt"),
           F.expr("percentile_approx(trans_amt, 0.5)").alias("median_amt"))
      .orderBy(F.desc("n_txn")).toPandas())
rc["txn_pct"] = (100 * rc.n_txn / rc.n_txn.sum()).round(2)
show(rc[["payment_rail", "category", "n_txn", "txn_pct", "sum_amt", "median_amt"]], n=40)

sub("deposit rows per calendar month  (gap check on the panel itself)")
mo = (dep_raw.groupBy(F.date_format("dt", "yyyy-MM").alias("month"))
      .agg(F.count("*").alias("n_rows"),
           F.countDistinct("dt").alias("n_dates"),
           F.countDistinct("acct_full_acct_id").alias("n_accts"))
      .orderBy("month").toPandas())
show(mo)

_fam_regex_share = pct(fam.loc[fam.regex_hit, "n_accts"].sum(), fam.n_accts.sum())
_month_gap = mo.n_dates.min() < 18
verdict(
    f"deposits {n_dep:,} rows / ~{dep_span.approx_accts[0]:,} accounts | payments {n_pay:,} rows",
    f"regex would keep {_fam_regex_share:.1f}% of accounts across "
    f"{int(fam.regex_hit.sum())} family values -- CONFIRM MANUALLY, do not trust the regex",
    ("month with <18 distinct dates present -- panel has calendar gaps, check the month table"
     if _month_gap else "every month carries >=18 distinct dates -- no calendar-level gaps"),
)

In [ ]:
# ── Apply the family filter and build the working views. ───────────────────
if DEPOSIT_FAMILY_KEEP:
    _fam_pred = "deposit_family IN (" + ", ".join(f"'{v}'" for v in DEPOSIT_FAMILY_KEEP) + ")"
else:
    _fam_pred = f"deposit_family RLIKE '{DEPOSIT_FAMILY_REGEX}'"
    print("!! DEPOSIT_FAMILY_KEEP is None -- falling back to the regex. Set it explicitly.")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW dep AS
SELECT * FROM dep_raw WHERE {_fam_pred}
""")
dep = spark.table("dep")

dep_mo = (dep
    .withColumn("month", F.date_format("dt", "yyyy-MM"))
    .withColumn("mo_idx", F.year("dt") * 12 + F.month("dt"))
    .groupBy("acct_full_acct_id", "month", "mo_idx")
    .agg(
        F.count("*").alias("n_rows"),
        F.countDistinct("dt").alias("n_days"),
        F.min("dt").alias("first_dt"),
        F.max("dt").alias("last_dt"),
        F.avg("balance").alias("bal_avg"),
        F.min("balance").alias("bal_min"),
        F.max("balance").alias("bal_max"),
        F.expr("max_by(balance, dt)").alias("bal_eom"),
        F.sum(F.when(F.col("balance") < 0, 1).otherwise(0)).alias("n_neg_days"),
        F.collect_set(nzs("acct_status")).alias("status_set"),
        F.max(nzs("cust_pwr_id")).alias("cust_pwr_id"),
        F.max(nzs("deposit_family")).alias("deposit_family"),
        F.max(F.to_date(F.col("closed_dt").cast("string"))).alias("closed_dt"),
        F.max(F.to_date(F.col("opened_dt").cast("string"))).alias("opened_dt"),
        F.max("avg_monthly_bal_1").alias("ambal_max"),
        F.countDistinct("avg_monthly_bal_1").alias("ambal_ndv"),
    )
    .persist())

_n_acct_mo = dep_mo.count()
print(f"{elapsed()} dep_mo built: {_n_acct_mo:,} account-months "
      f"({dep_mo.select('acct_full_acct_id').distinct().count():,} accounts)")

---
## OUTPUT 2 · Deposit panel grain and anomalies

Is it truly one row per account per business day? Where it isn't, the duplicate
groups are printed **transposed with every column**, differing fields marked
`*`, so the mechanism is visible in one glance rather than one screenshot per case.

In [ ]:
rule("OUTPUT 2 · DEPOSIT PANEL GRAIN")

_dep_n = dep.count()
_dep_keys = dep.select("acct_full_acct_id", "dt").distinct().count()

sub("grain")
grain = pd.DataFrame([{
    "rows": _dep_n,
    "distinct_(acct, dt)": _dep_keys,
    "excess_rows": _dep_n - _dep_keys,
    "excess_pct": round(pct(_dep_n - _dep_keys, _dep_n), 4),
}])
show(grain)

dup_keys = (dep.groupBy("acct_full_acct_id", "dt").agg(F.count("*").alias("n"))
            .filter(F.col("n") > 1).persist())
_n_dup_groups = dup_keys.count()

if _n_dup_groups:
    sub("duplicate multiplicity")
    show(dup_keys.groupBy("n").agg(F.count("*").alias("n_groups"))
         .orderBy("n").toPandas())

    sub("which columns actually differ inside a duplicate group")
    _other = [c for c in dep.columns if c not in ("acct_full_acct_id", "dt")]
    diff_agg = (dep.join(dup_keys.select("acct_full_acct_id", "dt"), ["acct_full_acct_id", "dt"])
                .groupBy("acct_full_acct_id", "dt")
                .agg(*[F.countDistinct(
                    F.coalesce(F.trim(F.col(c).cast("string")), F.lit("<NULL>"))).alias(c)
                    for c in _other]))
    varies = diff_agg.agg(*[F.sum(F.when(F.col(c) > 1, 1).otherwise(0)).alias(c)
                            for c in _other]).collect()[0].asDict()
    vdf = (pd.DataFrame({"column": list(varies), "groups_where_it_varies": list(varies.values())})
           .assign(pct_of_dup_groups=lambda d: (100 * d.groups_where_it_varies / _n_dup_groups).round(2))
           .sort_values("groups_where_it_varies", ascending=False))
    show(vdf)

    sub("three duplicate groups in full, transposed  ( * = column differs )")
    _samp_keys = dup_keys.orderBy(F.desc("n")).limit(3).select("acct_full_acct_id", "dt")
    _samp = (dep.join(_samp_keys, ["acct_full_acct_id", "dt"])
             .orderBy("acct_full_acct_id", "dt").toPandas())
    for (a, d), g in _samp.groupby(["acct_full_acct_id", "dt"], sort=False):
        t = g.astype(object).where(pd.notnull(g), "<NULL>").T
        t.columns = [f"row_{i + 1}" for i in range(t.shape[1])]
        t.insert(0, "", np.where(t.nunique(axis=1) > 1, "*", " "))
        print(f"\n  acct={a}  dt={d}  n_rows={t.shape[1] - 1}")
        print(t.to_string())
else:
    print("\n  no duplicate (acct, dt) groups.")

sub("business-day completeness, per account")
_cal = dep.select("dt").distinct()
_cal_mo = (_cal.withColumn("mo_idx", F.year("dt") * 12 + F.month("dt"))
           .groupBy("mo_idx").agg(F.countDistinct("dt").alias("cal_days")))
cov = (dep_mo.join(_cal_mo, "mo_idx")
       .withColumn("day_cov", F.col("n_days") / F.col("cal_days"))
       .agg(F.expr("percentile_approx(day_cov, array(0.01,0.10,0.25,0.50,0.75,0.90,1.0))").alias("q"),
            F.avg((F.col("day_cov") >= 0.999).cast("int")).alias("share_full_months"),
            F.avg((F.col("day_cov") < 0.5).cast("int")).alias("share_half_empty_months"))
       .toPandas())
qq = cov.q[0]
show(pd.DataFrame([{
    "day_cov_p01": qq[0], "p10": qq[1], "p25": qq[2], "p50": qq[3],
    "p75": qq[4], "p90": qq[5], "max": qq[6],
    "share_full_months": cov.share_full_months[0],
    "share_months_<50pct": cov.share_half_empty_months[0],
}]))

sub("account history depth and interior gaps")
span = (dep_mo.groupBy("acct_full_acct_id")
        .agg(F.count("*").alias("n_months"),
             F.min("mo_idx").alias("first_mo"),
             F.max("mo_idx").alias("last_mo"))
        .withColumn("span_months", F.col("last_mo") - F.col("first_mo") + 1)
        .withColumn("interior_gap_months", F.col("span_months") - F.col("n_months")))
show(span.agg(
    F.count("*").alias("n_accts"),
    F.expr("percentile_approx(n_months, array(0.1,0.5,0.9))").alias("n_months_q"),
    F.avg((F.col("interior_gap_months") > 0).cast("int")).alias("share_with_interior_gap"),
    F.expr("percentile_approx(interior_gap_months, 0.99)").alias("interior_gap_p99"),
).toPandas())

_dup_pct = pct(_dep_n - _dep_keys, _dep_n)
verdict(
    (f"grain is CLEAN: one row per (account, business day)" if _n_dup_groups == 0
     else f"{_n_dup_groups:,} duplicate (acct, dt) groups = {_dup_pct:.3f}% excess rows -- "
          f"see the 'varies' table for the splitting axis"),
    f"median month has {qq[3]:.2f} of that month's calendar business days present",
)

---
## OUTPUT 3 · Status semantics — as-of, or current-state?

The decisive question. If an account that closed in 2025-06 already carries
`acct_status='C'` on its 2024-03 row, the column is a **current-state stamp** and
can only ever be an outcome label — never a feature, never an as-of state.

In [ ]:
rule("OUTPUT 3 · acct_status / closed_dt SEMANTICS")

st = (dep
      .withColumn("status", nzs("acct_status"))
      .withColumn("cdt", F.to_date(F.col("closed_dt").cast("string")))
      .withColumn("odt", F.to_date(F.col("opened_dt").cast("string"))))

sub("value inventory")
show(st.groupBy("status").agg(
    F.count("*").alias("n_rows"),
    F.countDistinct("acct_full_acct_id").alias("n_accts"),
    F.avg(F.col("cdt").isNotNull().cast("int")).alias("share_rows_with_closed_dt"),
    F.expr("percentile_approx(balance, 0.5)").alias("median_bal"),
).orderBy(F.desc("n_rows")).toPandas())

acct_st = (st.groupBy("acct_full_acct_id").agg(
    F.countDistinct("status").alias("n_status_values"),
    F.countDistinct("cdt").alias("n_closed_dt_values"),
    F.min("dt").alias("first_dt"),
    F.max("dt").alias("last_dt"),
    F.max("cdt").alias("closed_dt"),
    F.min("odt").alias("opened_dt"),
    F.min(F.when(F.col("status") == "C", F.col("dt"))).alias("first_C_dt"),
    F.max(F.when(F.col("status") != "C", F.col("dt"))).alias("last_open_dt"),
    F.expr("min_by(status, dt)").alias("status_at_first_row"),
    F.expr("max_by(status, dt)").alias("status_at_last_row"),
).persist())
_n_acct = acct_st.count()

sub("per-account status behaviour")
beh = acct_st.agg(
    F.count("*").alias("n_accts"),
    F.avg((F.col("n_status_values") == 1).cast("int")).alias("share_single_status_value"),
    F.avg((F.col("n_closed_dt_values") > 1).cast("int")).alias("share_closed_dt_changes"),
    F.avg(F.col("closed_dt").isNotNull().cast("int")).alias("share_with_closed_dt"),
    F.avg((F.col("status_at_last_row") == "C").cast("int")).alias("share_C_at_last_row"),
    F.avg((F.col("status_at_first_row") == "C").cast("int")).alias("share_C_at_first_row"),
).toPandas()
show(beh)

sub("THE TEST -- accounts with a closed_dt: was 'C' already stamped before it?")
closers = acct_st.filter(F.col("closed_dt").isNotNull())
_n_closers = closers.count()
test = closers.agg(
    F.count("*").alias("n_accts_with_closed_dt"),
    F.avg((F.col("status_at_first_row") == "C").cast("int")).alias("share_C_on_FIRST_row"),
    F.avg(((F.col("status_at_first_row") == "C") & (F.col("first_dt") < F.col("closed_dt")))
          .cast("int")).alias("share_C_on_a_row_BEFORE_closure"),
    F.avg((F.col("first_C_dt") <= F.col("closed_dt")).cast("int")).alias("share_first_C_at_or_before_closed_dt"),
    F.expr("percentile_approx(DATEDIFF(first_C_dt, closed_dt), array(0.1,0.5,0.9))").alias("days_firstC_minus_closeddt_q"),
).toPandas()
show(test)

sub("does the account keep reporting after closure?")
after = closers.withColumn("days_last_row_after_close", F.datediff("last_dt", "closed_dt"))
show(after.agg(
    F.avg((F.col("days_last_row_after_close") > 0).cast("int")).alias("share_with_rows_after_closed_dt"),
    F.expr("percentile_approx(days_last_row_after_close, array(0.1,0.5,0.9,0.99))").alias("days_after_q"),
).toPandas())

_post = (st.join(closers.select("acct_full_acct_id", F.col("closed_dt").alias("cd")),
                 "acct_full_acct_id")
         .filter(F.col("dt") > F.col("cd")))
sub("balances on rows dated after closed_dt")
show(_post.agg(
    F.count("*").alias("n_rows_after_close"),
    F.avg((F.abs(F.col("balance")) <= ZERO_BAL_TOL).cast("int")).alias("share_near_zero"),
    F.expr("percentile_approx(balance, array(0.5,0.9,0.99))").alias("bal_q"),
).toPandas())

sub("opened_dt / closed_dt population")
show(st.agg(
    F.avg(F.col("odt").isNotNull().cast("int")).alias("share_rows_opened_dt"),
    F.avg(F.col("cdt").isNotNull().cast("int")).alias("share_rows_closed_dt"),
    F.avg((F.col("odt").isNotNull() & F.col("cdt").isNotNull()).cast("int")).alias("share_rows_both"),
    F.avg((F.col("odt").isNull() & F.col("cdt").isNull()).cast("int")).alias("share_rows_neither"),
).toPandas())

_pre_stamp = float(test.share_C_on_a_row_BEFORE_closure[0])
verdict(
    f"{_n_closers:,} of {_n_acct:,} accounts ({pct(_n_closers, _n_acct):.1f}%) carry a closed_dt in window",
    ("acct_status is a CURRENT-STATE STAMP -- 'C' appears on rows dated before closure. "
     "Usable as an outcome label only, never as a feature."
     if _pre_stamp > 0.20 else
     "acct_status looks AS-OF -- 'C' does not appear before closed_dt. Usable as a point-in-time state."),
    f"share of closers with 'C' stamped on a pre-closure row = {_pre_stamp:.3f}",
)

---
## OUTPUT 4 · Closure definitions vs. the real flag

Five candidates scored against `acct_status='C'` as ground truth: agreement,
and — for accounts both definitions catch — how many months early or late the
proxy fires. A proxy that agrees but fires **late** is worthless for early warning.

In [ ]:
rule("OUTPUT 4 · CLOSURE DEFINITIONS")

_end_mo = int(DATE_END[:4]) * 12 + int(DATE_END[5:7])
_panel_last_dt = dep.agg(F.max("dt")).collect()[0][0]

# monthly series with a contiguous month index; rangeBetween handles gaps correctly
w_ord  = W.partitionBy("acct_full_acct_id").orderBy("mo_idx")
w_3    = w_ord.rangeBetween(-2, 0)
w_6pri = w_ord.rangeBetween(-8, -3)
w_fwd  = w_ord.rangeBetween(0, ZERO_BAL_MONTHS - 1)

ser = (dep_mo
       .withColumn("avg3",   F.avg("bal_avg").over(w_3))
       .withColumn("n3",     F.count("bal_avg").over(w_3))
       .withColumn("avg6p",  F.avg("bal_avg").over(w_6pri))
       .withColumn("n6p",    F.count("bal_avg").over(w_6pri))
       .withColumn("empty",  (F.abs(F.col("bal_avg")) <= ZERO_BAL_TOL).cast("int"))
       .withColumn("empty_run_fwd", F.sum("empty").over(w_fwd))
       .withColumn("fire30", ((F.col("n3") == 3) & (F.col("n6p") == 6) &
                              (F.col("avg6p") > 0) &
                              (F.col("avg3") < DROP_RULE_RATIO * F.col("avg6p"))).cast("int"))
       .withColumn("fire_empty", (F.col("empty_run_fwd") == ZERO_BAL_MONTHS).cast("int")))

defs = (ser.groupBy("acct_full_acct_id").agg(
    F.max("mo_idx").alias("last_mo"),
    F.min(F.when(F.col("fire30") == 1, F.col("mo_idx"))).alias("mo_D5_drop30"),
    F.min(F.when(F.col("fire_empty") == 1, F.col("mo_idx"))).alias("mo_D4_empty"),
    F.max("closed_dt").alias("closed_dt"),
    F.min(F.when(F.array_contains(F.col("status_set"), "C"), F.col("mo_idx"))).alias("mo_D1_statusC"),
).withColumn("mo_D2_closeddt",
             F.when(F.col("closed_dt").isNotNull(),
                    F.year("closed_dt") * 12 + F.month("closed_dt")))
 .withColumn("mo_D3_disappear",
             F.when(F.col("last_mo") < _end_mo, F.col("last_mo") + 1))
 .persist())

DEFS = {"D1_statusC": "mo_D1_statusC", "D2_closed_dt": "mo_D2_closeddt",
        "D3_disappear": "mo_D3_disappear", "D4_empty_bal": "mo_D4_empty",
        "D5_drop30_rule": "mo_D5_drop30"}

sub("marginal counts")
_n_all = defs.count()
marg = defs.agg(*[F.sum(F.col(c).isNotNull().cast("int")).alias(k) for k, c in DEFS.items()]).toPandas().T
marg.columns = ["n_accts_flagged"]
marg["pct_of_book"] = (100 * marg.n_accts_flagged / _n_all).round(2)
show(marg.reset_index().rename(columns={"index": "definition"}))

sub("agreement with D1 (acct_status='C') -- confusion + timing")
rows = []
for k, c in DEFS.items():
    if k == "D1_statusC":
        continue
    a = defs.agg(
        F.sum((F.col("mo_D1_statusC").isNotNull() & F.col(c).isNotNull()).cast("int")).alias("both"),
        F.sum((F.col("mo_D1_statusC").isNotNull() & F.col(c).isNull()).cast("int")).alias("D1_only"),
        F.sum((F.col("mo_D1_statusC").isNull() & F.col(c).isNotNull()).cast("int")).alias("proxy_only"),
        F.expr(f"percentile_approx({c} - mo_D1_statusC, array(0.1,0.5,0.9))").alias("lag_q"),
        F.avg(F.when(F.col("mo_D1_statusC").isNotNull() & F.col(c).isNotNull(),
                     (F.abs(F.col(c) - F.col("mo_D1_statusC")) <= 1).cast("int"))).alias("share_within_1mo"),
    ).collect()[0]
    rows.append({
        "definition": k, "both": a["both"], "D1_only_missed": a["D1_only"],
        "proxy_only_false": a["proxy_only"],
        "precision_vs_D1": round(pct(a["both"], a["both"] + a["proxy_only"]), 2),
        "recall_vs_D1": round(pct(a["both"], a["both"] + a["D1_only"]), 2),
        "lag_p10": a["lag_q"][0], "lag_p50": a["lag_q"][1], "lag_p90": a["lag_q"][2],
        "share_within_1mo": round(float(a["share_within_1mo"] or 0), 3),
    })
agree = pd.DataFrame(rows)
show(agree)
print("\n  lag = proxy month - D1 month.  Negative = the proxy fires EARLIER than the status flag.")

sub("how many months of pre-closure history does each closer have?")
show(defs.filter(F.col("mo_D1_statusC").isNotNull())
     .join(dep_mo.groupBy("acct_full_acct_id").agg(F.min("mo_idx").alias("first_mo")),
           "acct_full_acct_id")
     .withColumn("hist_months", F.col("mo_D1_statusC") - F.col("first_mo"))
     .agg(F.count("*").alias("n_closers"),
          F.expr("percentile_approx(hist_months, array(0.1,0.25,0.5,0.75,0.9))").alias("hist_months_q"),
          F.avg((F.col("hist_months") >= 6).cast("int")).alias("share_ge_6mo"),
          F.avg((F.col("hist_months") >= 12).cast("int")).alias("share_ge_12mo"))
     .toPandas())

_best = agree.sort_values("lag_p50").iloc[0] if len(agree) else None
verdict(
    f"{int(marg.loc['D1_statusC', 'n_accts_flagged']):,} accounts hit acct_status='C' in window "
    f"({marg.loc['D1_statusC', 'pct_of_book']:.2f}% of the filtered book)",
    (f"earliest-firing proxy is {_best.definition} at median lag {_best.lag_p50:+.0f} months, "
     f"precision {_best.precision_vs_D1:.1f}% / recall {_best.recall_vs_D1:.1f}%"
     if _best is not None else "no proxy comparison available"),
)

---
## OUTPUT 5 · What is `avg_monthly_bal_1`?

Four candidates tested on a sampled cohort. The shape test comes first: if the
value **changes within a month** it is month-to-date; if it is constant it is a
completed-period average. Then exact match rates settle which period.

In [ ]:
rule("OUTPUT 5 · avg_monthly_bal_1 SEMANTICS")

_samp_accts = (dep.select("acct_full_acct_id").distinct()
               .filter(F.pmod(F.hash("acct_full_acct_id"), F.lit(101)) == F.lit(SEED % 101))
               .limit(SEMANTICS_SAMPLE_ACCTS))
d5 = (dep.join(F.broadcast(_samp_accts), "acct_full_acct_id")
      .select("acct_full_acct_id", "dt", "balance", "avg_monthly_bal_1")
      .withColumn("mo_idx", F.year("dt") * 12 + F.month("dt")))

sub("shape test -- does it vary within a calendar month?")
shape5 = (d5.groupBy("acct_full_acct_id", "mo_idx")
          .agg(F.countDistinct("avg_monthly_bal_1").alias("ndv"),
               F.count("*").alias("n_days"))
          .filter(F.col("n_days") >= 5)
          .agg(F.count("*").alias("n_acct_months"),
               F.avg((F.col("ndv") <= 1).cast("int")).alias("share_constant_in_month"),
               F.expr("percentile_approx(ndv, array(0.5,0.9))").alias("ndv_q"))
          .toPandas())
show(shape5)

w_acct = W.partitionBy("acct_full_acct_id").orderBy("mo_idx")
w_mtd  = W.partitionBy("acct_full_acct_id", "mo_idx").orderBy("dt").rowsBetween(W.unboundedPreceding, 0)
w_30d  = (W.partitionBy("acct_full_acct_id")
          .orderBy(F.col("dt").cast("timestamp").cast("long"))
          .rangeBetween(-29 * 86400, 0))

mo_avg = d5.groupBy("acct_full_acct_id", "mo_idx").agg(F.avg("balance").alias("cur_full_mo"))
mo_avg = mo_avg.withColumn("prior_full_mo", F.lag("cur_full_mo").over(w_acct))

cand = (d5.join(mo_avg, ["acct_full_acct_id", "mo_idx"])
        .withColumn("mtd_running", F.avg("balance").over(w_mtd))
        .withColumn("trailing_30d", F.avg("balance").over(w_30d)))

CANDS = ["mtd_running", "cur_full_mo", "prior_full_mo", "trailing_30d"]
sub("match rate against each candidate  (tolerance: max($1, 0.1%))")
tol = F.greatest(F.lit(1.0), F.abs(F.col("avg_monthly_bal_1")) * 0.001)
m = cand.filter(F.col("avg_monthly_bal_1").isNotNull()).agg(
    F.count("*").alias("n_rows_scored"),
    *[F.avg((F.abs(F.col("avg_monthly_bal_1") - F.col(c)) <= tol).cast("int")).alias(c) for c in CANDS],
    *[F.expr(f"percentile_approx(abs(avg_monthly_bal_1 - {c}), 0.5)").alias(f"med_abs_err__{c}")
      for c in CANDS],
).toPandas()
res = pd.DataFrame({
    "candidate": CANDS,
    "exact_match_rate": [float(m[c][0]) for c in CANDS],
    "median_abs_error": [float(m[f"med_abs_err__{c}"][0]) for c in CANDS],
}).sort_values("exact_match_rate", ascending=False)
print(f"  rows scored: {int(m.n_rows_scored[0]):,}   (sample of {SEMANTICS_SAMPLE_ACCTS:,} accounts)")
show(res)

sub("balance column sanity -- negatives, zeros, scale")
show(dep.agg(
    F.avg((F.col("balance") < 0).cast("int")).alias("share_rows_negative"),
    F.avg((F.col("balance") == 0).cast("int")).alias("share_rows_exact_zero"),
    F.expr("percentile_approx(balance, array(0.01,0.25,0.5,0.75,0.99))").alias("bal_q"),
    F.countDistinct(F.when(F.col("balance") < 0, F.col("acct_full_acct_id"))).alias("n_accts_ever_negative"),
).toPandas())

_win = res.iloc[0]
verdict(
    ("value is CONSTANT within a calendar month -- a completed-period average, not MTD"
     if float(shape5.share_constant_in_month[0]) > 0.9 else
     "value CHANGES within a calendar month -- month-to-date accumulation"),
    f"best candidate: {_win.candidate} at {100*_win.exact_match_rate:.1f}% exact match "
    f"(median abs error ${_win.median_abs_error:,.2f})",
    ("no candidate exceeds 80% -- it is none of these four; add it to the open-questions list"
     if _win.exact_match_rate < 0.8 else "settled -- use this definition downstream"),
)

---
## OUTPUT 6 · Payments profile and integrity

Leg classification, amount sanity, counterparty-name and FI-name coverage, and
a verification of the one-row-per-transaction claim. Also a first read on the
exact-name match that gates `same_name_outflow_flag`.

In [ ]:
rule("OUTPUT 6 · PAYMENTS PROFILE")

p = (pay_raw
     .withColumn("mp",   nzs("mdm_id_pays"))
     .withColumn("mr",   nzs("mdm_id_receives"))
     .withColumn("ap",   nzs("pnc_dep_acct_pays"))
     .withColumn("ar",   nzs("pnc_dep_acct_receives"))
     .withColumn("cnp",  nzs("customer_name_pays"))
     .withColumn("cnr",  nzs("customer_name_receives"))
     .withColumn("cpty", nzs("cpty_name"))
     .withColumn("fin",  nzs("cpty_fin_entity_name"))
     .withColumn("cpid", nzs("unq_cpty_acct_id"))
     .withColumn("leg", F.when(F.col("mp").isNotNull() & F.col("mr").isNotNull(), "internal")
                        .when(F.col("mp").isNull()    & F.col("mr").isNotNull(), "inbound")
                        .when(F.col("mp").isNotNull() & F.col("mr").isNull(),    "outbound")
                        .otherwise("orphan"))
     .persist())

sub("leg classification")
show(p.groupBy("leg").agg(
    F.count("*").alias("n_txn"),
    F.sum("trans_amt").alias("sum_amt"),
    F.avg(F.col("cpid").isNotNull().cast("int")).alias("share_w_cpty_acct_id"),
    F.avg(F.col("cpty").isNotNull().cast("int")).alias("share_w_cpty_name"),
    F.avg(F.col("fin").isNotNull().cast("int")).alias("share_w_fin_entity"),
    F.avg(F.col("ap").isNotNull().cast("int")).alias("share_w_acct_pays"),
    F.avg(F.col("ar").isNotNull().cast("int")).alias("share_w_acct_receives"),
).orderBy(F.desc("n_txn")).toPandas())

sub("trans_amt sanity")
show(p.agg(
    F.avg((F.col("trans_amt") < 0).cast("int")).alias("share_negative"),
    F.avg((F.col("trans_amt") == 0).cast("int")).alias("share_zero"),
    F.avg(F.col("trans_amt").isNull().cast("int")).alias("share_null"),
    F.expr("percentile_approx(trans_amt, array(0.01,0.5,0.99,0.999))").alias("amt_q"),
    F.max("trans_amt").alias("amt_max"),
).toPandas())

sub("one row per transaction -- verification")
_n_txn = p.count()
_n_tid = p.select("trans_id").distinct().count()
_econ = (p.groupBy("dt", "trans_amt", "ap", "ar", "cpid")
         .agg(F.countDistinct("trans_id").alias("n_ids"))
         .filter(F.col("n_ids") > 1))
_n_econ = _econ.count()
_econ_int = (p.filter(F.col("leg") == "internal")
             .groupBy("dt", "trans_amt", "ap", "ar")
             .agg(F.countDistinct("trans_id").alias("n_ids"))
             .filter(F.col("n_ids") > 1).count())
show(pd.DataFrame([{
    "rows": _n_txn, "distinct_trans_id": _n_tid,
    "id_collisions": _n_txn - _n_tid,
    "same_(dt,amt,acct_pays,acct_recv,cpty_id)_multi_id_groups": _n_econ,
    "  of which internal legs": _econ_int,
    "pct_of_txn_in_such_groups": round(pct(_n_econ * 2, _n_txn), 4),
}]))
print("  Note: same-day/same-amount groups are not proof of double-booking -- a genuine")
print("  repeated payment looks identical. Treat a large internal-leg count as the alarm.")

sub("counterparty name and FI coverage, by direction  (outbound = customer pays external)")
show(p.filter(F.col("leg").isin("inbound", "outbound")).groupBy("leg", nzs("payment_rail").alias("rail"))
     .agg(F.count("*").alias("n_txn"),
          F.sum("trans_amt").alias("sum_amt"),
          F.avg(F.col("cpty").isNotNull().cast("int")).alias("cpty_name_cov"),
          F.avg(F.col("fin").isNotNull().cast("int")).alias("fin_entity_cov"))
     .orderBy(F.desc("n_txn")).toPandas(), n=25)

sub("value-weighted coverage on OUTBOUND -- this is what gates Group A")
_ob = p.filter(F.col("leg") == "outbound")
show(_ob.agg(
    F.sum("trans_amt").alias("total_outflow"),
    (F.sum(F.when(F.col("cpty").isNotNull(), F.col("trans_amt")).otherwise(0)) /
     F.sum("trans_amt")).alias("amt_share_name_classifiable"),
    (F.sum(F.when(F.col("fin").isNotNull(), F.col("trans_amt")).otherwise(0)) /
     F.sum("trans_amt")).alias("amt_share_fi_classifiable"),
).toPandas())

sub("first read on same-name outflow  (exact match after case/space normalisation)")
_norm = lambda c: F.upper(F.regexp_replace(F.col(c), r"[^A-Za-z0-9]", ""))
_sn = (_ob.filter(F.col("cpty").isNotNull() & F.col("cnp").isNotNull())
       .withColumn("same_name", (_norm("cpty") == _norm("cnp")).cast("int")))
show(_sn.agg(
    F.count("*").alias("n_scored_txn"),
    F.avg("same_name").alias("txn_share_same_name"),
    (F.sum(F.when(F.col("same_name") == 1, F.col("trans_amt")).otherwise(0)) /
     F.sum("trans_amt")).alias("amt_share_same_name"),
    F.countDistinct(F.when(F.col("same_name") == 1, F.col("mp"))).alias("n_customers_with_any"),
).toPandas())

sub("top cpty_fin_entity_name by outbound value  (the fi_destination_flag universe)")
show(_ob.filter(F.col("fin").isNotNull()).groupBy("fin")
     .agg(F.count("*").alias("n_txn"), F.sum("trans_amt").alias("sum_amt"),
          F.countDistinct("mp").alias("n_customers"))
     .orderBy(F.desc("sum_amt")).toPandas(), n=25)

_orphan = p.filter(F.col("leg") == "orphan").count()
verdict(
    f"{_n_txn:,} transactions, {_n_tid:,} distinct trans_id, {_orphan:,} orphan legs (no PNC side)",
    ("one row per transaction CONFIRMED -- absolute counts and dollar totals are valid"
     if _econ_int == 0 else
     f"{_econ_int:,} internal-leg groups share (dt, amt, both accounts) under different trans_ids "
     "-- inspect before trusting internal dollar totals"),
    "cpty_fin_entity_name replaces the maintained FI name list from brief S12",
)

---
## OUTPUT 7 · Payments ↔ deposits join coverage

How much of the payment book actually lands on an account we hold deposit data
for. This number is the ceiling on everything the study can see.

In [ ]:
rule("OUTPUT 7 · JOIN COVERAGE")

dep_accts = dep.select(F.col("acct_full_acct_id").alias("acct")).distinct().persist()
_n_dep_accts = dep_accts.count()

sub("account id present when the mdm_id is?")
show(p.agg(
    F.avg(F.when(F.col("mp").isNotNull(), F.col("ap").isNotNull().cast("int"))).alias("pays: acct given mdm"),
    F.avg(F.when(F.col("mr").isNotNull(), F.col("ar").isNotNull().cast("int"))).alias("recv: acct given mdm"),
    F.avg(F.when(F.col("ap").isNotNull(), F.col("mp").isNotNull().cast("int"))).alias("pays: mdm given acct"),
    F.avg(F.when(F.col("ar").isNotNull(), F.col("mr").isNotNull().cast("int"))).alias("recv: mdm given acct"),
).toPandas())

legs = (p.select(F.col("dt"), F.col("trans_amt"), F.col("leg"),
                 F.explode(F.array(
                     F.struct(F.lit("pays").alias("side"), F.col("ap").alias("acct")),
                     F.struct(F.lit("recv").alias("side"), F.col("ar").alias("acct")))).alias("e"))
        .select("dt", "trans_amt", "leg", "e.side", "e.acct")
        .filter(F.col("acct").isNotNull()))

joined = legs.join(dep_accts.withColumn("in_dep", F.lit(1)), "acct", "left")

sub("join rate of payment account ids into the deposit table")
show(joined.groupBy("leg", "side").agg(
    F.count("*").alias("n_legs"),
    F.avg(F.coalesce(F.col("in_dep"), F.lit(0))).alias("leg_join_rate"),
    (F.sum(F.when(F.col("in_dep") == 1, F.col("trans_amt")).otherwise(0)) /
     F.sum("trans_amt")).alias("amt_join_rate"),
).orderBy("leg", "side").toPandas())

sub("join rate by month  (drift means an extract-boundary problem, not a data problem)")
show(joined.groupBy(F.date_format("dt", "yyyy-MM").alias("month"))
     .agg(F.count("*").alias("n_legs"),
          F.avg(F.coalesce(F.col("in_dep"), F.lit(0))).alias("leg_join_rate"))
     .orderBy("month").toPandas())

sub("the anchor population -- deposit accounts with any payment activity")
pay_accts = legs.select(F.col("acct")).distinct()
_n_anchor = dep_accts.join(pay_accts, "acct").count()
show(pd.DataFrame([{
    "deposit_accts_in_scope": _n_dep_accts,
    "with_>=1_payment_leg": _n_anchor,
    "coverage_pct": round(pct(_n_anchor, _n_dep_accts), 2),
}]))

sub("cust_pwr_id <-> mdm_id cardinality")
_map = (p.select(F.col("ap").alias("acct"), F.col("mp").alias("mdm"))
        .union(p.select(F.col("ar").alias("acct"), F.col("mr").alias("mdm")))
        .filter(F.col("acct").isNotNull() & F.col("mdm").isNotNull()).distinct()
        .join(dep.select(F.col("acct_full_acct_id").alias("acct"),
                         nzs("cust_pwr_id").alias("cpid")).distinct(), "acct")
        .select("cpid", "mdm").distinct().persist())
card = _map.groupBy("cpid").agg(F.countDistinct("mdm").alias("n_mdm"))
card2 = _map.groupBy("mdm").agg(F.countDistinct("cpid").alias("n_cpid"))
show(pd.DataFrame([{
    "linked_pairs": _map.count(),
    "cust_pwr_id_with_>1_mdm": card.filter("n_mdm > 1").count(),
    "mdm_id_with_>1_cust_pwr_id": card2.filter("n_cpid > 1").count(),
    "share_cpid_one_to_one": round(float(card.agg(F.avg((F.col("n_mdm") == 1).cast("int"))).collect()[0][0]), 4),
}]))

sub("accounts per customer, in the deposit table")
show(dep.groupBy(nzs("cust_pwr_id").alias("cpid"))
     .agg(F.countDistinct("acct_full_acct_id").alias("n_accts"))
     .agg(F.count("*").alias("n_customers"),
          F.expr("percentile_approx(n_accts, array(0.5,0.9,0.99))").alias("accts_per_cust_q"),
          F.max("n_accts").alias("max_accts"))
     .toPandas())

verdict(
    f"{_n_anchor:,} of {_n_dep_accts:,} in-scope deposit accounts "
    f"({pct(_n_anchor, _n_dep_accts):.1f}%) carry at least one payment leg",
    "a low value-weighted join rate on outbound legs is the binding constraint on Group A",
)

---
## OUTPUT 8 · Episode feasibility

Do we have enough observed departures, with enough history in front of them, to
run the study — and does our own book reproduce the benchmark shape from §2 of
the brief? Median index, not mean: one whale otherwise writes the curve.

In [ ]:
rule("OUTPUT 8 · EPISODE FEASIBILITY")

closers_mo = defs.filter(F.col("mo_D1_statusC").isNotNull()).select(
    "acct_full_acct_id", F.col("mo_D1_statusC").alias("t0"))

sub("closures per calendar month")
_cm = (closers_mo.withColumn("month", F.concat_ws("-",
        F.lpad(((F.col("t0") - 1) / 12).cast("int").cast("string"), 4, "0"),
        F.lpad((((F.col("t0") - 1) % 12) + 1).cast("string"), 2, "0")))
       .groupBy("month").agg(F.count("*").alias("n_closures")).orderBy("month").toPandas())
show(_cm)

case_curve = (dep_mo.join(closers_mo, "acct_full_acct_id")
              .withColumn("rel", F.col("mo_idx") - F.col("t0"))
              .filter(F.col("rel").between(-EVENT_WINDOW, 0)))
anchor_c = (case_curve.filter(F.col("rel") == -EVENT_WINDOW)
            .select("acct_full_acct_id", F.col("bal_avg").alias("base"))
            .filter(F.col("base") > ZERO_BAL_TOL))
case_idx = (case_curve.join(anchor_c, "acct_full_acct_id")
            .withColumn("idx", 100 * F.col("bal_avg") / F.col("base"))
            .groupBy("rel").agg(F.expr("percentile_approx(idx, 0.5)").alias("cases_median_index"),
                                F.count("*").alias("n_cases")))

ctrl_accts = (defs.filter(F.col("mo_D1_statusC").isNull() & F.col("mo_D2_closeddt").isNull())
              .select("acct_full_acct_id")
              .filter(F.pmod(F.hash("acct_full_acct_id"), F.lit(101)) == F.lit(SEED % 101))
              .limit(CONTROL_SAMPLE)
              .withColumn("t0", F.lit(_end_mo)))
ctrl_curve = (dep_mo.join(ctrl_accts, "acct_full_acct_id")
              .withColumn("rel", F.col("mo_idx") - F.col("t0"))
              .filter(F.col("rel").between(-EVENT_WINDOW, 0)))
anchor_k = (ctrl_curve.filter(F.col("rel") == -EVENT_WINDOW)
            .select("acct_full_acct_id", F.col("bal_avg").alias("base"))
            .filter(F.col("base") > ZERO_BAL_TOL))
ctrl_idx = (ctrl_curve.join(anchor_k, "acct_full_acct_id")
            .withColumn("idx", 100 * F.col("bal_avg") / F.col("base"))
            .groupBy("rel").agg(F.expr("percentile_approx(idx, 0.5)").alias("controls_median_index"),
                                F.count("*").alias("n_controls")))

curve = (case_idx.join(ctrl_idx, "rel", "outer").orderBy("rel").toPandas())
sub(f"event-time median balance index (month -{EVENT_WINDOW} = 100), cases vs controls")
show(curve)

sub("how much of the book survives the study filters")
_surv = (defs.join(dep_mo.groupBy("acct_full_acct_id").agg(F.min("mo_idx").alias("first_mo"),
                                                           F.avg("bal_avg").alias("mean_bal")),
                   "acct_full_acct_id")
         .withColumn("is_case", F.col("mo_D1_statusC").isNotNull().cast("int"))
         .withColumn("hist", F.coalesce(F.col("mo_D1_statusC"), F.lit(_end_mo)) - F.col("first_mo")))
show(_surv.agg(
    F.count("*").alias("accounts"),
    F.sum("is_case").alias("cases"),
    F.sum(F.when((F.col("is_case") == 1) & (F.col("hist") >= 6), 1).otherwise(0)).alias("cases_6mo_hist"),
    F.sum(F.when((F.col("is_case") == 1) & (F.col("hist") >= 12), 1).otherwise(0)).alias("cases_12mo_hist"),
    F.sum(F.when((F.col("is_case") == 1) & (F.col("hist") >= 12) &
                 (F.col("mean_bal") >= 25000), 1).otherwise(0)).alias("cases_12mo_and_25k"),
).toPandas())

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 4.2))
    ax.plot(curve.rel, curve.cases_median_index, marker="o", lw=2, label="closers (cases)")
    ax.plot(curve.rel, curve.controls_median_index, marker="s", lw=2, label="never-closed (controls)")
    ax.axhline(100, color="grey", lw=0.8, ls="--")
    ax.set_xlabel("months relative to closure"); ax.set_ylabel(f"median balance index (m-{EVENT_WINDOW}=100)")
    ax.set_title("Balance trajectory into closure — PKG attrition feasibility")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
except Exception as e:  # noqa: BLE001
    print(f"  (plot skipped: {e})")

_c = curve.dropna(subset=["cases_median_index"])
_onset = None
if len(_c):
    _below = _c[_c.cases_median_index < 90]
    _onset = int(_below.rel.min()) if len(_below) else None
verdict(
    f"{int(_surv.agg(F.sum('is_case')).collect()[0][0]):,} closure episodes in window",
    (f"median closer balance first drops below 90 at month {_onset} relative to closure "
     f"-- that is the observed onset lead, against the benchmark's 8-9 months"
     if _onset is not None else
     "median closer balance never falls below 90 before closure -- closures are NOT balance-preceded, "
     "which changes the whole premise and is itself the finding"),
)

In [ ]:
rule("RUN COMPLETE")
print(f"  elapsed {elapsed()}")
print("""
  Open questions this notebook does NOT answer, for the follow-up list:
    - whether closed accounts are replaced by a new account for the same
      cust_pwr_id (product switch vs. true attrition) -- needs the customer-level pass
    - counterparty-side account continuity (does unq_cpty_acct_id persist across banks)
    - intra-month timing of balance moves -- deferred with the daily grain
""")
for _v in ["dep_mo", "dup_keys", "acct_st", "defs", "p", "dep_accts", "_map"]:
    try:
        eval(_v).unpersist()
    except Exception:  # noqa: BLE001
        pass